# 📊 Model Evaluation & Comparison

This notebook compares all three models side-by-side with comprehensive evaluations.

**Prerequisites**: Run all model notebooks first:
1. `01_preprocessing.ipynb`
2. `02_train_decision_tree.ipynb`
3. `03_train_xgboost.ipynb`
4. `04_train_hgnn.ipynb`

In [ ]:
import sys, os
import warnings
warnings.filterwarnings('ignore')
import pickle

# Ensure PROJECT_P is on the path
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Project modules
from src.config import *
from src.utils import get_logger, set_plot_style

logger = get_logger('Evaluation')
print('✅ Imports successful')

## 1. Load All Model Results

In [ ]:
# Load all model results
results = {}
models = ['decision_tree', 'xgboost', 'hgnn']
labels = ['Decision Tree', 'XGBoost', 'HGNN-ATT-TD']

for model_name, label in zip(models, labels):
    results_file = os.path.join(MODEL_DIR, f'{model_name}_results.pkl')
    
    if not os.path.exists(results_file):
        print(f'⚠️  Missing: {results_file}')
        continue
    
    with open(results_file, 'rb') as f:
        results[label] = pickle.load(f)
    
    print(f'✅ Loaded {label}')

print(f'\n📊 Loaded {len(results)} model results')

## 2. Create Comparison Table

In [ ]:
# Create comparison dataframe
comparison_data = []

for model_name, model_results in results.items():
    metrics = model_results['metrics']
    comparison_data.append({
        'Model': model_name,
        'Accuracy': f"{metrics['Accuracy']:.4f}",
        'Precision': f"{metrics['Precision']:.4f}",
        'Recall': f"{metrics['Recall']:.4f}",
        'F1-Score': f"{metrics['F1-Score']:.4f}",
        'ROC-AUC': f"{metrics['ROC-AUC']:.4f}",
    })

comp_df = pd.DataFrame(comparison_data)

print('\n' + '='*80)
print('           FINAL MODEL COMPARISON')
print('='*80)
print(comp_df.to_string(index=False))
print('='*80)

## 3. Generate Comparison Plots

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve, confusion_matrix, roc_auc_score

set_plot_style()

# Load test data
preprocess_file = os.path.join(MODEL_DIR, 'preprocessed_data.pkl')
with open(preprocess_file, 'rb') as f:
    prep = pickle.load(f)

y_test = prep['y_test']

# 3a. Precision-Recall Curves
fig, ax = plt.subplots(figsize=(10, 6))

for model_name, model_results in results.items():
    y_pred_proba = model_results['y_pred_proba']
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    ax.plot(recall, precision, label=f'{model_name}', linewidth=2.5, alpha=0.8)

ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax.set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, '01_precision_recall_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Precision-Recall curves saved')

In [ ]:
# 3b. ROC Curves
fig, ax = plt.subplots(figsize=(10, 6))

for model_name, model_results in results.items():
    y_pred_proba = model_results['y_pred_proba']
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)
    ax.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.4f})', linewidth=2.5, alpha=0.8)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, '02_roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ ROC curves saved')

In [ ]:
# 3c. Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for idx, (model_name, model_results) in enumerate(results.items()):
    y_pred = model_results['y_pred']
    cm = confusion_matrix(y_test, y_pred)
    
    # Plot
    im = axes[idx].imshow(cm, cmap='Blues', aspect='auto')
    axes[idx].set_title(f'{model_name}\nConfusion Matrix', fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_xticks([0, 1])
    axes[idx].set_yticks([0, 1])
    axes[idx].set_xticklabels(['Legit', 'Fraud'])
    axes[idx].set_yticklabels(['Legit', 'Fraud'])
    
    # Add text
    for i in range(2):
        for j in range(2):
            text = axes[idx].text(j, i, cm[i, j], ha='center', va='center',
                                color='white' if cm[i, j] > cm.max()/2 else 'black',
                                fontsize=12, fontweight='bold')
    
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, '03_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrices saved')

In [ ]:
# 3d. Metrics Comparison Bar Chart
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
model_names = list(results.keys())

for idx, metric in enumerate(metrics_list):
    values = [results[m]['metrics'][metric] for m in model_names]
    colors = plt.cm.Set2(np.linspace(0, 1, len(model_names)))
    
    bars = axes[idx].bar(model_names, values, color=colors, edgecolor='black', linewidth=1.5)
    axes[idx].set_ylabel(metric, fontweight='bold')
    axes[idx].set_title(f'{metric} Comparison', fontweight='bold')
    axes[idx].set_ylim([0, 1])
    axes[idx].tick_params(axis='x', rotation=15)
    
    # Add value labels
    for bar, val in zip(bars, values):
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                       f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    axes[idx].grid(axis='y', alpha=0.3)

# Remove extra subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, '05_comparison_table.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Metrics comparison saved')

## 4. Save Comparison Summary

In [ ]:
# Save comparison as CSV
comp_csv = comp_df.copy()
comp_csv.to_csv(os.path.join(EVAL_DIR, 'model_comparison.csv'), index=False)

print('\n📊 Evaluation Summary:')
print(f'  ✅ Precision-Recall curves')
print(f'  ✅ ROC curves')
print(f'  ✅ Confusion matrices')
print(f'  ✅ Metrics comparison')
print(f'  ✅ Comparison CSV saved')
print(f'\n✅ ALL EVALUATIONS COMPLETE!')